In [0]:
from pyspark.sql.functions import *

In [0]:
silver_table = 'workspace.taxi.silver_trips'
fact_table = 'workspace.taxi.fact_trips'

In [0]:
df = spark.table(silver_table)

In [0]:
df.display()

In [0]:
fact_df = df.withColumn('trip_date',to_date('pickup_ts'))\
    .withColumn('revenue',col('fare_amount'))\
    .select(
        monotonically_increasing_id().alias('trip_sk'),
        'vendorID','pickup_ts','dropoff_ts','trip_date','passenger_count','trip_distance','revenue','PULocationID','DOLocationID'
    )

In [0]:
fact_df.display()

In [0]:
fact_df.write.format('delta').mode('overwrite').saveAsTable(fact_table)

print(f"Gold Fact Trips complete. Rows: {spark.table(fact_table).count()}")

In [0]:
vendor_df = df.select('vendorID').distinct()\
    .withColumn('vendor_name',when(col('vendorID')==1,"Creative Mobile technologies")
                .when(col('vendorID')==2,"VeriFone Inc.")
                .otherwise('Unknown'))

In [0]:
vendor_df.display()

In [0]:
vendor_df.write.format('delta').mode('overwrite').saveAsTable('workspace.taxi.dim_vendors')

In [0]:
location_df = spark.read.csv('/Volumes/workspace/taxi/raw_data/taxi_zone_lookup.csv')

In [0]:
location_df.display()

In [0]:
location_df.write.format('delta').mode('overwrite').saveAsTable('workspace.taxi.dim_locations')